In [30]:
from youtube_transcript_api import YouTubeTranscriptApi
import yt_dlp
from pathlib import Path
import pandas as pd
import time
from sentence_transformers import SentenceTransformer
from sentence_transformers import util

In [4]:
video_id = "1zF1wxyB5Qk"

ytt_api = YouTubeTranscriptApi()
transcript = ytt_api.fetch(video_id)

In [5]:
df = pd.DataFrame(transcript.to_raw_data())

df.head(10)

,text,start,duration
0,[Music],0.000,14.320
1,hello and welcome to perfect person the,12.480,5.280
2,call and advice show about how to become,14.320,5.600
3,the perfect version of yourself i'm,17.760,6.320
4,miles bonson your webby award winner and,19.920,6.240
5,all-around perfect person i've done all,24.080,4.080
6,sorts of things in the podcast world and,26.160,5.279
7,now i'm sort of bravely,28.160,6.480
8,i'm bravely setting out in in a world,31.439,5.521
9,that no one's really touched before and,34.640,4.800


In [6]:
print(f"Total transcript segments: {len(df)}")
print(f"Columns: {df.columns.tolist()}")

print("\nSample transcript:")
print(" ".join(df["text"].head(20)))

Total transcript segments: 1667
Columns: ['text', 'start', 'duration']

Sample transcript:
[Music] hello and welcome to perfect person the call and advice show about how to become the perfect version of yourself i'm miles bonson your webby award winner and all-around perfect person i've done all sorts of things in the podcast world and now i'm sort of bravely i'm bravely setting out in in a world that no one's really touched before and it's podcasts no one has made a podcast no one's made podcasts you want a webby for one but that was different well as a webby award winner i feel like i get sort of how podcasts are done in a way that a lot of people don't you know how the sausage is made and you're here to make your own sausage i know how the sausage is made and i'm ready to sort of tackle it with the charm and


In [ ]:
# Extract Video IDs

playlist_url = "https://www.youtube.com/playlist?list=PL_3nBFXOTXucTnnnctJr1HH4Vs-N8FhS1"

options = {
    "extract_flat": True,
    "skip_download": True,
    "quiet": True
}

with yt_dlp.YoutubeDL(options) as ydl:
    info = ydl.extract_info(playlist_url, download=False)

episodes = []

for video in info.get("entries", []):
    if not video or not video.get("id"):
        continue

    episodes.append({
        "video_id": video["id"],
        "title": video.get("title"),
        "url": f"https://www.youtube.com/watch?v={video['id']}"
    })

episodes_df = pd.DataFrame(episodes)

episodes_df.head()


,video_id,title,url
0,9-y9-GW2ISM,attempted murder at my local craft night (w/ B...,https://www.youtube.com/watch?v=9-y9-GW2ISM
1,OBaGlNyIuyo,my boss is sabotaging our birthday parties (w/...,https://www.youtube.com/watch?v=OBaGlNyIuyo
2,AEt7mCD1njw,my sister's being haunted by a demon (w/ Eric ...,https://www.youtube.com/watch?v=AEt7mCD1njw
3,kbrl7dkMxhI,accidentally dating my ex's brother (w/ Will W...,https://www.youtube.com/watch?v=kbrl7dkMxhI
4,S6Qks9yHHCU,devastatingly sent my whole ass to my boss (w/...,https://www.youtube.com/watch?v=S6Qks9yHHCU


In [13]:
print(f"Total videos found: {len(episodes_df)}")
print(f"Duplicate IDs: {episodes_df['video_id'].duplicated().sum()}")

episodes_df.head(10)

episodes_df.to_csv("data/episodes.csv", index=False)

Total videos found: 225
Duplicate IDs: 0


In [15]:
ytt_api = YouTubeTranscriptApi()

output_dir = Path("data/transcripts")
output_dir.mkdir(parents=True, exist_ok=True)

def get_transcript(video_id, title):
    output_path = output_dir / f"{video_id}.json"

    # Skip episodes we've already downloaded
    if output_path.exists():
        return "skipped"

    # Fetch transcript
    transcript = ytt_api.fetch(video_id)

    # Convert to DataFrame
    df = pd.DataFrame(transcript.to_raw_data())

    # Add episode metadata
    df["video_id"] = video_id
    df["episode_title"] = title
    df["url"] = f"https://www.youtube.com/watch?v={video_id}"

    # Save to JSON
    df.to_json(
        output_path,
        orient="records",
        indent=2,
        force_ascii=False
    )

    return "success"

In [22]:
transcript_files = list(Path("data/transcripts").glob("*.json"))

all_transcripts = pd.concat(
    [pd.read_json(file) for file in transcript_files],
    ignore_index=True
)

all_transcripts = all_transcripts.sort_values(
    ["video_id", "start"]
).reset_index(drop=True)

print(f"Episodes: {all_transcripts['video_id'].nunique()}")
print(f"Transcript segments: {len(all_transcripts)}")

all_transcripts.head()

Episodes: 5
Transcript segments: 11658


,text,start,duration,video_id,episode_title,url
0,This is a Headgum podcast.,1.920,5.339,9-y9-GW2ISM,attempted murder at my local craft night (w/ B...,https://www.youtube.com/watch?v=9-y9-GW2ISM
1,[music],5.239,2.020,9-y9-GW2ISM,attempted murder at my local craft night (w/ B...,https://www.youtube.com/watch?v=9-y9-GW2ISM
2,"Hello and welcome to Perfect Person, the",17.440,2.720,9-y9-GW2ISM,attempted murder at my local craft night (w/ B...,https://www.youtube.com/watch?v=9-y9-GW2ISM
3,show where I'm perfect and you're of,18.880,4.399,9-y9-GW2ISM,attempted murder at my local craft night (w/ B...,https://www.youtube.com/watch?v=9-y9-GW2ISM
4,course a person.,20.160,7.680,9-y9-GW2ISM,attempted murder at my local craft night (w/ B...,https://www.youtube.com/watch?v=9-y9-GW2ISM


In [23]:
def chunk_transcript(df, chunk_size=150, overlap=30):
    chunks = []

    # Process each episode separately
    for video_id, episode in df.groupby("video_id", sort=False):

        episode = episode.sort_values("start")

        words = []

        # Preserve each word's original timestamp
        for _, row in episode.iterrows():
            for word in row["text"].split():
                words.append({
                    "word": word,
                    "start": row["start"],
                    "end": row["start"] + row["duration"]
                })

        step = chunk_size - overlap

        for i in range(0, len(words), step):
            window = words[i:i + chunk_size]

            if not window:
                continue

            chunks.append({
                "video_id": video_id,
                "episode_title": episode["episode_title"].iloc[0],
                "text": " ".join(w["word"] for w in window),
                "start": window[0]["start"],
                "end": window[-1]["end"],
                "chunk_id": f"{video_id}_{i}"
            })

    return pd.DataFrame(chunks)

In [24]:
chunks_df = chunk_transcript(all_transcripts)

print(f"Created {len(chunks_df)} chunks!")

chunks_df.head()

Created 610 chunks!


,video_id,episode_title,text,start,end,chunk_id
0,9-y9-GW2ISM,attempted murder at my local craft night (w/ B...,This is a Headgum podcast. [music] Hello and w...,1.920,64.720,9-y9-GW2ISM_0
1,9-y9-GW2ISM,attempted murder at my local craft night (w/ B...,you into Maybe they [laughter] did. There was ...,54.320,97.200,9-y9-GW2ISM_120
2,9-y9-GW2ISM,attempted murder at my local craft night (w/ B...,I think that exhaustion will lead to um like s...,84.159,122.159,9-y9-GW2ISM_240
3,9-y9-GW2ISM,attempted murder at my local craft night (w/ B...,here. >> Thanks for giving us the space. This ...,113.600,158.400,9-y9-GW2ISM_360
4,9-y9-GW2ISM,attempted murder at my local craft night (w/ B...,"the same. >> Yes. >> Holy [ __ ] Well, guys, w...",146.560,192.239,9-y9-GW2ISM_480


In [25]:
for _, row in chunks_df.head(3).iterrows():
    print(f"\n{'=' * 50}")
    print(f"Episode: {row['episode_title']}")
    print(f"Start: {row['start']:.1f} seconds")
    print(f"End: {row['end']:.1f} seconds")
    print(f"\n{row['text']}")


Episode: attempted murder at my local craft night (w/ Business Casual) | Perfect Person Ep. 224
Start: 1.9 seconds
End: 64.7 seconds

This is a Headgum podcast. [music] Hello and welcome to Perfect Person, the show where I'm perfect and you're of course a person. >> Oh boom, pizza for lunch is back. Yeah, I actually had an upside down pepperoni slice. >> What makes it upside down? >> It's fkaca. >> It's [laughter] I think upside down means the cheese is underneath. So it's basically on the No, sorry. That's a That's a regular pizza. The cheese is underneath. So it's bread, cheese, sauce, pepperoni on top of the sauce. >> Oh, >> it was way messier. I don't know why they do that. Sloppy. >> It was way sloppy. >> It sounds like they messed up and kind of tricked you into Maybe they [laughter] did. There was a restaurant where the guy runs around screaming and sells you pizza. >> That's like when you're 20 and you're super stone

Episode: attempted murder at my local craft night (w/ Busin

In [27]:
output_path = Path("data/chunks.json")

chunks_df.to_json(
    output_path,
    orient="records",
    indent=2,
    force_ascii=False
)

print("Chunks saved!")

Chunks saved!


In [28]:
print(f"Total chunks: {len(chunks_df)}")

Total chunks: 610


In [31]:
model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    chunks_df["text"].tolist(),
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embedding shape:", embeddings.shape)

Batches: 100%|██████████| 20/20 [00:02<00:00,  8.80it/s]

Embedding shape: (610, 384)


In [32]:
from sentence_transformers import util

query = "advice about friendship problems"

query_embedding = model.encode(
    query,
    normalize_embeddings=True
)

scores = util.cos_sim(query_embedding, embeddings)[0]

top_indices = scores.argsort(descending=True)[:5]

for idx in top_indices:
    row = chunks_df.iloc[int(idx)]

    print(f"\nEpisode: {row['episode_title']}")
    print(f"Similarity: {scores[idx]:.3f}")
    print(f"Timestamp: {row['start']:.0f} seconds")
    print(f"\n{row['text'][:500]}...")
    print("-" * 50)


Episode: attempted murder at my local craft night (w/ Business Casual) | Perfect Person Ep. 224
Similarity: 0.516
Timestamp: 2476 seconds

folks off to the side, but don't stop being open to new people. >> And what's better than a group? >> No, for sure. >> Yeah. >> And I love I love this friend group, so definitely want to stay friends. >> I'm a good girl. I can handle it. >> Yeah. Well, >> well, and [snorts] Jeremy, how would you handle this situation? Is this >> No, I think I think you're nailing it. Don't because you don't like I think you would want to limit like your like one-on- ones and your like twoon- ones. Like I think l...
--------------------------------------------------

Episode: accidentally dating my ex's brother (w/ Will Witwer) | Perfect Person Ep. 221
Similarity: 0.468
Timestamp: 2664 seconds

I was just on FaceTime with him. So you Okay, so you don't need advice anymore, but you just have a juicy story. >> Did you Wait, no. Did you confront them? >> No, I have not

In [33]:
def search_podcast(query, top_k=5):
    query_embedding = model.encode(
        query,
        normalize_embeddings=True
    )

    scores = util.cos_sim(query_embedding, embeddings)[0]
    top_indices = scores.argsort(descending=True)[:top_k]

    for idx in top_indices:
        row = chunks_df.iloc[int(idx)]
        timestamp = int(row["start"])

        url = (
            f"https://www.youtube.com/watch?"
            f"v={row['video_id']}&t={timestamp}s"
        )

        print(f"\n🎧 {row['episode_title']}")
        print(f"Similarity: {scores[idx]:.3f}")
        print(f"Timestamp: {timestamp // 60}:{timestamp % 60:02d}")
        print(f"Listen: {url}")
        print(f"\n{row['text'][:500]}...")
        print("-" * 60)

In [34]:
search_podcast("advice about friendship problems")


🎧 attempted murder at my local craft night (w/ Business Casual) | Perfect Person Ep. 224
Similarity: 0.516
Timestamp: 41:15
Listen: https://www.youtube.com/watch?v=9-y9-GW2ISM&t=2475s

folks off to the side, but don't stop being open to new people. >> And what's better than a group? >> No, for sure. >> Yeah. >> And I love I love this friend group, so definitely want to stay friends. >> I'm a good girl. I can handle it. >> Yeah. Well, >> well, and [snorts] Jeremy, how would you handle this situation? Is this >> No, I think I think you're nailing it. Don't because you don't like I think you would want to limit like your like one-on- ones and your like twoon- ones. Like I think l...
------------------------------------------------------------

🎧 accidentally dating my ex's brother (w/ Will Witwer) | Perfect Person Ep. 221
Similarity: 0.468
Timestamp: 44:24
Listen: https://www.youtube.com/watch?v=kbrl7dkMxhI&t=2664s

I was just on FaceTime with him. So you Okay, so you don't need advice a

In [35]:
def format_timestamp(seconds):
    minutes, seconds = divmod(int(seconds), 60)
    return f"{minutes:02d}:{seconds:02d}"


def search_podcast(query, top_k=5, min_gap=180):

    # 1. Embed the user's question
    query_embedding = model.encode(
        query,
        normalize_embeddings=True
    )

    # 2. Calculate similarity with every chunk
    scores = util.cos_sim(query_embedding, embeddings)[0]

    # 3. Sort all chunks by similarity
    ranked_indices = scores.argsort(descending=True)

    results = []

    # 4. Keep only distinct search hits
    for idx in ranked_indices:
        row = chunks_df.iloc[int(idx)]

        start = float(row["start"])

        # Check whether we've already found
        # a nearby passage in this episode
        too_close = any(
            result["video_id"] == row["video_id"]
            and abs(result["start"] - start) < min_gap
            for result in results
        )

        if too_close:
            continue

        results.append({
            "episode": row["episode_title"],
            "video_id": row["video_id"],
            "start": start,
            "timestamp": format_timestamp(start),
            "similarity": round(float(scores[idx]), 3),
            "preview": row["text"],
            "url": (
                f"https://www.youtube.com/watch?"
                f"v={row['video_id']}&t={int(start)}s"
            )
        })

        if len(results) >= top_k:
            break

    return pd.DataFrame(results)

In [36]:
results = search_podcast(
    "advice about friendship problems"
)

results[["episode", "timestamp", "similarity", "url"]]

,episode,timestamp,similarity,url
0,attempted murder at my local craft night (w/ B...,41:15,0.516,https://www.youtube.com/watch?v=9-y9-GW2ISM&t=...
1,accidentally dating my ex's brother (w/ Will W...,44:24,0.468,https://www.youtube.com/watch?v=kbrl7dkMxhI&t=...
2,attempted murder at my local craft night (w/ B...,36:29,0.349,https://www.youtube.com/watch?v=9-y9-GW2ISM&t=...
3,my boss is sabotaging our birthday parties (w/...,22:09,0.334,https://www.youtube.com/watch?v=OBaGlNyIuyo&t=...
4,attempted murder at my local craft night (w/ B...,32:17,0.326,https://www.youtube.com/watch?v=9-y9-GW2ISM&t=...


In [37]:
from IPython.display import display, Markdown

def display_results(results):

    if results.empty:
        display(Markdown("No results found."))
        return

    for i, row in results.iterrows():

        display(Markdown(
            f"### {i + 1}. {row['episode']}\n"
            f"**Timestamp:** {row['timestamp']}  \n"
            f"**Similarity:** {row['similarity']:.3f}  \n"
            f"[▶ Listen on YouTube]({row['url']})"
        ))

        print(row["preview"][:700])
        print("\n" + "─" * 60)

In [38]:
results = search_podcast(
    "advice about navigating friendships"
)

display_results(results)

### 1. attempted murder at my local craft night (w/ Business Casual) | Perfect Person Ep. 224
**Timestamp:** 40:45  
**Similarity:** 0.469  
[▶ Listen on YouTube](https://www.youtube.com/watch?v=9-y9-GW2ISM&t=2445s)

not saving himself for marriage. Um or or you'll find somebody that's more aligned. >> I think that's actually beautiful. That's like a good note like just let it be and like sometimes you lust after this friend but like let your friend group blossom. You're also new to this city. That's a different lens to this whole thing is that you're in a new city. It's easy to like first people that you really connect with to get deeper into that. >> Yeah. I was going to say like it's only been three Yeah. It's like these are your this is your friend group that you currently got, but it's only been 3 months. Like don't necessarily throw these folks off to the side, but don't stop being open to new peop

────────────────────────────────────────────────────────────


### 2. accidentally dating my ex's brother (w/ Will Witwer) | Perfect Person Ep. 221
**Timestamp:** 44:24  
**Similarity:** 0.422  
[▶ Listen on YouTube](https://www.youtube.com/watch?v=kbrl7dkMxhI&t=2664s)

I was just on FaceTime with him. So you Okay, so you don't need advice anymore, but you just have a juicy story. >> Did you Wait, no. Did you confront them? >> No, I have not confronted that. >> Okay, so yeah, perfect. So wait, what? So this is a really close friend. >> Um, so it was actually like a situationship from earlier in the summer. >> A situation from earlier in the summer. >> What does that mean? A situationship in the summer. >> We need a song on situation. Yeah, we went on a couple dates and then he was too busy and then we just started becoming friends again. Now we're just friends. >> You went on a couple dates and then he started getting too busy and then you guys hang out as 

────────────────────────────────────────────────────────────


### 3. my boss is sabotaging our birthday parties (w/ Stef Dag) | Perfect Person Ep. 223
**Timestamp:** 22:09  
**Similarity:** 0.401  
[▶ Listen on YouTube](https://www.youtube.com/watch?v=OBaGlNyIuyo&t=1329s)

you could rise to the top of the Midwest. I think they need they need a cool Midwest it girl. >> You can go viral from anywhere now. >> You could do Hey, >> you know what I mean? >> Internet exists everywhere. >> Internet exists everywhere. People are going viral for just doing whatever. >> LA's overrun. I think like you know like I think you're probably a Midwest 11. [laughter] >> You're a Midwest 15 out of 10. Thank you. >> Yeah. And I think that you're gonna absolutely do it. I mean, what why don't you also reconnect? It doesn't seem like it's >> this cousin's fault that you guys aren't friends anymore. Your mom had beef. >> No, it's absolutely not anyone's fault. >> Reach out to be like,

────────────────────────────────────────────────────────────


### 4. devastatingly sent my whole ass to my boss (w/ Anna Garcia) | Perfect Person Ep. 220
**Timestamp:** 33:04  
**Similarity:** 0.367  
[▶ Listen on YouTube](https://www.youtube.com/watch?v=S6Qks9yHHCU&t=1984s)

I used to be a server and so whenever people would say that to me I'll be like oh yeah of course I want to get it for you but sometimes I'm just now realizing that >> I don't apply that to my own life >> this is a huge realization >> this is actually a big realization what are you studying in school by the way >> I'm studying Well, I don't really [clears throat] know yet. I'm doing like a politics economics thing, but we'll see what I stick with. >> Okay. Well, honestly, I think that what you need to do is um go ask the >> I think you should go Wait, so what do you order in person every time or is there ever a Postmates sort of situation? >> It's over like a GrubHub thing. >> Oh, and you ord

────────────────────────────────────────────────────────────


### 5. attempted murder at my local craft night (w/ Business Casual) | Perfect Person Ep. 224
**Timestamp:** 36:29  
**Similarity:** 0.352  
[▶ Listen on YouTube](https://www.youtube.com/watch?v=9-y9-GW2ISM&t=2189s)

do is be like, "Hey, maybe we don't hang out, you know, for a little bit or whatever." >> Yeah. Yeah. Do you think that that's an option? Do you want to stay friends with him even if you >> I just mean in a temporary way? Like and it cool off a cool off like a like Yeah. 8 days of temporary cooling off. >> Yeah. It's like right now when you see him when you're in your words ovulating you >> which [laughter] is you say that that way cuz you don't believe in it. >> Yeah. You said that with a lot of disdain Miles. Yeah. >> Whatever that is. Sorry. Okay. And I got to play that. >> Yeah. Okay. [laughter] >> Um Yeah. Sure. >> Misogyny alert. >> Okay. Great. >> So, and you know what? That one's on 

────────────────────────────────────────────────────────────


In [47]:
results = search_podcast(
    "advice about friendship problems",
    top_k=5
)

results[["episode", "timestamp", "similarity", "url"]]
display_results(results)

### 1. attempted murder at my local craft night (w/ Business Casual) | Perfect Person Ep. 224
**Timestamp:** 41:15  
**Similarity:** 0.516  
[▶ Listen on YouTube](https://www.youtube.com/watch?v=9-y9-GW2ISM&t=2475s)

folks off to the side, but don't stop being open to new people. >> And what's better than a group? >> No, for sure. >> Yeah. >> And I love I love this friend group, so definitely want to stay friends. >> I'm a good girl. I can handle it. >> Yeah. Well, >> well, and [snorts] Jeremy, how would you handle this situation? Is this >> No, I think I think you're nailing it. Don't because you don't like I think you would want to limit like your like one-on- ones and your like twoon- ones. Like I think like that's something you can just kind of quietly to yourself. >> This guy isn't going to sleep with her. He's definitely not going to have a threesome with her. What are you >> doing? [laughter] Hang

────────────────────────────────────────────────────────────


### 2. accidentally dating my ex's brother (w/ Will Witwer) | Perfect Person Ep. 221
**Timestamp:** 44:24  
**Similarity:** 0.468  
[▶ Listen on YouTube](https://www.youtube.com/watch?v=kbrl7dkMxhI&t=2664s)

I was just on FaceTime with him. So you Okay, so you don't need advice anymore, but you just have a juicy story. >> Did you Wait, no. Did you confront them? >> No, I have not confronted that. >> Okay, so yeah, perfect. So wait, what? So this is a really close friend. >> Um, so it was actually like a situationship from earlier in the summer. >> A situation from earlier in the summer. >> What does that mean? A situationship in the summer. >> We need a song on situation. Yeah, we went on a couple dates and then he was too busy and then we just started becoming friends again. Now we're just friends. >> You went on a couple dates and then he started getting too busy and then you guys hang out as 

────────────────────────────────────────────────────────────


### 3. attempted murder at my local craft night (w/ Business Casual) | Perfect Person Ep. 224
**Timestamp:** 36:29  
**Similarity:** 0.349  
[▶ Listen on YouTube](https://www.youtube.com/watch?v=9-y9-GW2ISM&t=2189s)

do is be like, "Hey, maybe we don't hang out, you know, for a little bit or whatever." >> Yeah. Yeah. Do you think that that's an option? Do you want to stay friends with him even if you >> I just mean in a temporary way? Like and it cool off a cool off like a like Yeah. 8 days of temporary cooling off. >> Yeah. It's like right now when you see him when you're in your words ovulating you >> which [laughter] is you say that that way cuz you don't believe in it. >> Yeah. You said that with a lot of disdain Miles. Yeah. >> Whatever that is. Sorry. Okay. And I got to play that. >> Yeah. Okay. [laughter] >> Um Yeah. Sure. >> Misogyny alert. >> Okay. Great. >> So, and you know what? That one's on 

────────────────────────────────────────────────────────────


### 4. my boss is sabotaging our birthday parties (w/ Stef Dag) | Perfect Person Ep. 223
**Timestamp:** 22:09  
**Similarity:** 0.334  
[▶ Listen on YouTube](https://www.youtube.com/watch?v=OBaGlNyIuyo&t=1329s)

you could rise to the top of the Midwest. I think they need they need a cool Midwest it girl. >> You can go viral from anywhere now. >> You could do Hey, >> you know what I mean? >> Internet exists everywhere. >> Internet exists everywhere. People are going viral for just doing whatever. >> LA's overrun. I think like you know like I think you're probably a Midwest 11. [laughter] >> You're a Midwest 15 out of 10. Thank you. >> Yeah. And I think that you're gonna absolutely do it. I mean, what why don't you also reconnect? It doesn't seem like it's >> this cousin's fault that you guys aren't friends anymore. Your mom had beef. >> No, it's absolutely not anyone's fault. >> Reach out to be like,

────────────────────────────────────────────────────────────


### 5. attempted murder at my local craft night (w/ Business Casual) | Perfect Person Ep. 224
**Timestamp:** 32:17  
**Similarity:** 0.326  
[▶ Listen on YouTube](https://www.youtube.com/watch?v=9-y9-GW2ISM&t=1937s)

Okay. In the landscape. Good. Fresh air helps clear the head. That's right. >> I recently moved cities about six months ago and I kind of got added to this friend group probably about three months ago. >> Okay. >> You know, most of them are men and there's like me and two other girls. And [clears throat] I don't know how this conversation conversation keeps coming up, but for some reason one of the friends just like they always talk about his dick size. [screaming] And I'm just like, okay, cool. They talk about like legendary photo that >> they have a legendary photo. So, let's just say it involves a bed sheet and uh waking up in the morning. >> Okay. Yeah, I got a picture of his bed under a

────────────────────────────────────────────────────────────


In [48]:
import numpy as np

np.save("data/embeddings.npy", embeddings)